# v110_m3_features — M3's feature groups in the two-stage model, dense blocking, GPU stage 1

| Field | Value |
|---|---|
| **Version** | `v110_m3_features` |
| **Plan group** | C2 (M3's feature groups), A5 (blocking), D4 (GPU stage 1), C5 (two-stage) |
| **Parent version** | v107 (public 0.966: v104's two-stage, rule tuned on the tight mock) |
| **Author** | M1 rajaguru2004 (feature groups: M3, PR #8) |
| **Date** | 2026-09-26 |
| **Status** | shortlisted |

v107 (public 0.966, est_public 0.9659) is the two-stage model on v101's 53 features and the
default blocking. v110 rebuilds it with everything merged since:

* **M3's four feature groups** (PR #8; v040 gained +0.0026 local F0.5 over v001 as a single
  stage): `idf` (8 columns), `token_freq` (4), `ctx_idf` (5) and `address_extra` (6), 76
  features in all.
* **Dense-pool blocking:** v105's `B6` (sim-first cap, bigger budgets) plus v109's name +
  address-number exact pass, cap 120 (`db1f33c4`): mock candidate recall 0.9677 → 0.9829.
* **Stage 1 retrained on the GPU** (XGBoost) at train-fold density with the 76 features:
  v106's design, which ran out of GPU memory at 18M rows; `stage1.fit_stage1` now sizes its
  row cap by the feature count.
* **Stage 2** as in v104 plus rival features; the rule tuned on the tight mock as in v107.

```
train-fold S1 absent from the mock ─► blocking vs train-fold pool ─► 76 features → disk chunks
    ─► XGBoost QuantileDMatrix on the GPU (streamed) ─► stage 1
mock fold ─► stage 1 on every pair ─► filter + competition + anchor + rival features
    ─► stage 2 (GPU, cross-fitted on the mock's fit entities) ─► rule on mock tune ─► mock val
```

## 1. Hypothesis

* **Change vs parent (v107):** features (M3's four groups), blocking (B6 + name+number, cap
  120) and stage 1 (XGBoost on the GPU, trained on train-fold entities absent from the mock,
  at train-fold density). Stage 2 gains rival features. The filter, the competition and
  anchor features and the tight rule tuning are v107's.
* **Why:** v107 loses true pairs at blocking (0.9677 of them survive at test density, v105)
  and at the stage-1 filter (0.9644, v104); v109's blocking buys most of that back. M3's
  groups (idf-weighted token overlap, pool token frequency, context idf, address containment
  and numbers) separate the near-duplicates that the tight metric punishes (false merges
  ×1.45).
* **Check:** est_public on the mock val entities against v107's 0.9659 (public 0.966); stage 1
  alone (threshold rule) against v103; ablations of stage 2 without M3's columns and without
  the rival columns.
* **Discard if:** est_public does not beat v107 by more than 0.001.

## 2. Setup

Configurations: the pipeline (v101's feature groups + M3's `idf`, `token_freq`, `ctx_idf`,
`address_extra`; blocking B6 + name+number exact pass with cap 120, v109's adopted
`db1f33c4`), stage 1 (XGBoost on the GPU, 127 leaves, learning rate 0.1, early stopping on a 5 %
id-hash slice of its own entities; 110k training entities, ~9M pairs: the rows × features a
4 GB card trains on) and the two-stage settings of v104 with rival features.

In [1]:
import json
import shutil
import subprocess
import sys
import time
from dataclasses import asdict, replace

import numpy as np
import pandas as pd

from entity_resolution import config as C
from entity_resolution.blocking import TopKSpec
from entity_resolution.data import isin
from entity_resolution.decision import DecisionRule, apply_rule, decide, tune_expected
from entity_resolution.evaluate import blocking_report, error_samples
from entity_resolution.features import DEFAULT_GROUPS
from entity_resolution.mock import FP_WEIGHT, PUBLIC_OFFSET, build_mock, target_shape
from entity_resolution.model import MatcherParams
from entity_resolution.pipeline import (
    Fitted, PipelineConfig, learn_token_map, mock_scores, peak_rss_gb, tune_mock,
)
from entity_resolution.split import load_fold
from entity_resolution.stage1 import absent_from_mock, clean, fit_stage1, write_chunks
from entity_resolution.tracking import log_result, timed
from entity_resolution.trainset import inner_split, sample_s1
from entity_resolution.twostage import (
    TwoStage, TwoStageConfig, fit_stage2, mock_scored, mock_stage1, run_test_two_stage,
)

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.max_columns", 30)

EXP_DIR = C.EXPERIMENTS / "v110_m3_features"
ARTIFACTS = EXP_DIR / "artifacts"
M3_GROUPS = ("idf", "token_freq", "ctx_idf", "address_extra")    # PR #8 (M3), as in v040
# B6 (v105/v106): bigger word top-k and caps, sim-first cap; v109 adds the name + address-number
# exact pass (groups of at most 100) and a cap of 120 candidates per S1 entity
B6 = replace(PipelineConfig().blocking,
             name_addr_word=replace(PipelineConfig().blocking.name_addr_word, top_k=50),
             max_per_s1=100, exact_max_group=200,
             addr_char=TopKSpec("addr_norm", "word", (1, 2), top_k=10, min_sim=0.3,
                                max_df=0.01, max_df_abs=10_000),
             cap_order="sim_first")
cfg = PipelineConfig(feature_groups=(*DEFAULT_GROUPS, "frequency", *M3_GROUPS),
                     model=MatcherParams(n_estimators=4000),
                     blocking=replace(B6, name_num_max_group=100, max_per_s1=120))
assert B6.key() == "b54f2a3d" and cfg.blocking.key() == "db1f33c4"   # v106's, v109's adopted
S1_PARAMS = MatcherParams(backend="xgb", device="cuda", num_leaves=127, learning_rate=0.1,
                          n_estimators=3000, early_stopping=50)
N_STAGE1 = 110_000  # entities: ~9M pairs x 76 features, what the 4 GB card trains on
tcfg = TwoStageConfig(floor=0.01, max_cands=16, cohesion=False, rivals=True,
                      model=MatcherParams(backend="xgb", device="cuda", n_estimators=4000))
# v107 = v104 with the rule tuned on the tight mock (public 0.966): the est_public to beat
v107 = json.loads((C.EXPERIMENTS / "v107_tight_rule" / "metrics.json").read_text())
EST_TO_BEAT = v107["metrics"]["comparison"]["v107 tight rule"]["est_public"]
v103 = json.loads((C.EXPERIMENTS / "v103_mock_rule" / "metrics.json").read_text())
STAGE1_CACHE = (cfg.cache_dir / "stage1" /
                f"v110_{cfg.blocking.key()}_f{tcfg.floor}_k{tcfg.max_cands}_a{int(tcfg.anchors)}"
                f"_c{int(tcfg.cohesion)}_r{int(tcfg.rivals)}")
timings: dict[str, float] = {}
t_start = time.time()
print("blocking", cfg.blocking.key(), "|", len(cfg.feature_groups), "feature groups",
      "| v107 est_public to beat", round(EST_TO_BEAT, 4), "| v107 mock F0.5", v107["mock_f05"],
      "| v103", v103["mock_f05"])

blocking db1f33c4 | 13 feature groups | v107 est_public to beat 0.9659 | v107 mock F0.5 0.9745 | v103 0.9704


## 3. Data

The mock fold (as in v103–v107; its blocking under this configuration is cached or built on
first use) and the stage-1 training entities: train-fold S1 absent from the mock, capped at
`N_STAGE1` by id hash (`sample_s1`), about 9M candidate pairs.

In [2]:
with timed("load", timings):
    train = load_fold("train", columns=[C.COUNTRY])
    val = load_fold("val", columns=[C.COUNTRY])
    fit_fold, tune_fold = inner_split(train)
    fit_sample = sample_s1(fit_fold.s1, cfg.n_fit_s1)[C.ENTITY_ID]
    mock = build_mock(train, val, tune_fold.s1[C.ENTITY_ID], fit_sample, target_shape())
    # the transliteration token map learned from train-fold pairs (cached; identical to the
    # map of v101, so this notebook needs no earlier version's artifacts)
    token_map = learn_token_map(cfg, train)
del val, fit_fold, tune_fold, fit_sample
absent = absent_from_mock(mock, train)
s1_ids = pd.Index(sample_s1(train.s1[isin(train.s1[C.ENTITY_ID], absent)], N_STAGE1)[C.ENTITY_ID])
print(f"absent from the mock: {len(absent):,}; stage-1 training entities: {len(s1_ids):,}")
train.s1[isin(train.s1[C.ENTITY_ID], s1_ids)][C.COUNTRY].value_counts()

absent from the mock: 732,845; stage-1 training entities: 212,941


country
US       133562
India     79379
Name: count, dtype: int64

## 4. Method

### 4.1 Stage 1 v2 on the GPU

Blocking of the training entities against the train-fold pool (cached), features chunk by
chunk to disk, then XGBoost on the GPU reading the chunks through a `DataIter`
(`stage1.fit_stage1`). The chunks are deleted once the model is saved.

In [3]:
work = cfg.cache_dir / "stage1_chunks" / f"v110_{cfg.blocking.key()}"
t0 = time.time()
manifest = (json.loads((work / "manifest.json").read_text()) if (work / "manifest.json").exists()
            else write_chunks(cfg, train, s1_ids, token_map, work, timings=timings))
timings["stage1_set_seconds"] = round(time.time() - t0, 2)
print(f"training set {timings['stage1_set_seconds']:.0f} s: {manifest['rows']:,} pairs of "
      f"{manifest['entities']:,} entities, positive rate {manifest['positives'] / manifest['rows']:.3f}")
t0 = time.time()
s1_model = fit_stage1(manifest, S1_PARAMS)
timings["stage1_fit_seconds"] = round(time.time() - t0, 2)
stage1 = Fitted(s1_model, DecisionRule(), pd.DataFrame({"f_beta": [np.nan]}), cfg, token_map,
                {"stage1": "xgb gpu", "fit_info": s1_model.fit_info_})
stage1.save(ARTIFACTS / "stage1_v2")
clean(work)
print(json.dumps(s1_model.fit_info_, indent=1))
s1_model.importance().head(15).rename("gain share").to_frame()

training set 947 s: 14,900,006 pairs of 212,941 entities, positive rate 0.049


{
 "rows": 6985490,
 "positive_rate": 0.048513201940992505,
 "best_iteration": 396,
 "tune_logloss": 0.005671949216980335,
 "tune_auc": 0.9998419283750657,
 "fit_seconds": 207.82,
 "entity_share_used": 0.49266557342325906
}


,gain share
feature,
ctx_gap_addr,0.272439
ad_token_set,0.130421
squash_ratio,0.103787
ctx_rank_addr,0.096990
sim_name_addr_word,0.067397
ctx_rank_idf_addr,0.046458
core_jw,0.032062
core_token_set,0.025503
num_contain_l,0.021474


### 4.2 Stage 1 alone on the mock

The stage-1 pass keeps every pair with p1 ≥ 0.01 among its entity's 16 best: all pairs any
threshold rule could keep. The threshold rule on `p1`, tuned on the mock tune entities, gives
stage 1's single-stage mock F0.5, comparable with v103 (v101 + mock rule).

In [4]:
t0 = time.time()
outs = mock_stage1(cfg, stage1, mock, tcfg, timings=timings, cache_dir=STAGE1_CACHE / "mock")
timings["stage1_mock_seconds"] = round(time.time() - t0, 2)
from entity_resolution.decision import one_to_one_filter
keep_ids = pd.Index(mock.fold.s1[C.ENTITY_ID][mock.role.isin(["tune", "val"]).to_numpy()])
p1_scored = []
for o in outs.values():
    s = o.pairs.assign(prob=o.X["p1"].to_numpy())
    s = one_to_one_filter(s)
    p1_scored.append(s[isin(s[C.S1_ID], keep_ids)])
p1_scored = pd.concat(p1_scored, ignore_index=True)
rule1, _ = tune_mock(p1_scored, mock, cfg.grid, fp_weight=FP_WEIGHT)
single = mock_scores(p1_scored, mock, rule1)
del p1_scored
kept = pd.concat([o.pairs for o in outs.values()], ignore_index=True)
filter_report = pd.DataFrame({r: blocking_report(kept, mock.part(r)) for r in ("tune", "val")}).T
print(f"stage-1 pass {timings['stage1_mock_seconds']:.0f} s; rule {rule1}")
display(single[["f_beta", "f_tight", "est_public", "f_beta_singletons", "pair_precision",
                "pair_recall"]])
filter_report[["pair_recall", "entity_recall", "ceiling_f_beta", "candidates_mean", "candidates_p95"]]

/home/suryaguru/StudioProjects/aws/business_entity_resolution/.venv/lib64/python3.12/site-packages/xgboost/core.py:774: UserWarning: [15:15:47] WARNING: /workspace/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


stage-1 pass 4054 s; rule DecisionRule(tau_abs=0.78, tau_rel=0.7, tau_single=0.83, max_matches=11, one_to_one=True)


,f_beta,f_tight,est_public,f_beta_singletons,pair_precision,pair_recall
all,0.977068,0.975334,0.968134,0.981998,0.995534,0.942973
India,0.974840,0.972773,0.965573,0.978732,0.994627,0.937999
US,0.979474,0.978098,0.970898,0.985525,0.996506,0.948349


,pair_recall,entity_recall,ceiling_f_beta,candidates_mean,candidates_p95
tune,0.978509,0.998378,0.993190,4.400792,8.0
val,0.978817,0.998414,0.993291,4.399334,8.0


### 4.3 Stage 2 on top, as in v104

Cross-fitted stage 2 on the mock's fit entities (GPU), stage-2 probabilities with the
pool-side 1-to-1 across every present entity, the rule tuned on the mock tune entities for
the tight metric (threshold or expected-F0.5 rule, whichever scores higher there).

In [5]:
t0 = time.time()
models, fit_info = fit_stage2(outs, mock, tcfg)
scored, report = mock_scored(outs, models, mock, tcfg)
scored.to_parquet(ARTIFACTS / "mock_scored.parquet", index=False)
rule_t, table_t = tune_mock(scored, mock, cfg.grid, fp_weight=FP_WEIGHT)
tune_part = mock.part("tune")
rows_t = scored[isin(scored[C.S1_ID], pd.Index(tune_part.s1[C.ENTITY_ID]))]
rule_e, table_e = tune_expected(rows_t, tune_part.s1[C.ENTITY_ID], tune_part.pairs,
                                gammas=(0.7, 0.85, 1.0, 1.2, 1.5, 2.0),
                                misses=(0.0, 0.05, 0.1, 0.2, 0.4), fp_weight=FP_WEIGHT)
rule, table = ((rule_e, table_e) if table_e["f_beta"].max() > table_t["f_beta"].max()
               else (rule_t, table_t))
timings["stage2_seconds"] = round(time.time() - t0, 2)
res = mock_scores(scored, mock, rule)
print(f"stage 2 {timings['stage2_seconds']:.0f} s; rule {rule} (tight-tuned)")
importance = pd.concat([m.importance() for m in models], axis=1).mean(axis=1)
display(importance.sort_values(ascending=False).head(15).rename("gain share").to_frame())
res[["f_beta", "f_tight", "est_public", "f_beta_singletons", "pair_precision", "pair_recall",
     "entities"]]

stage 2 426 s; rule ExpectedRule(gamma=1.5, miss=0.1, max_matches=11, one_to_one=True) (tight-tuned)


,gain share
feature,
pool_gap,0.689176
p1,0.178514
s1_gap,0.039600
s1_p1_sum,0.005999
pool_p1_sum,0.005676
pool_best_other,0.004605
pool_rank,0.002780
sim_name_addr_word,0.002553
s1_best_other,0.002181


,f_beta,f_tight,est_public,f_beta_singletons,pair_precision,pair_recall,entities
all,0.981650,0.980331,0.973131,0.984667,0.996882,0.951820,340084.0
India,0.980236,0.978848,0.971648,0.983469,0.996746,0.947229,176522.0
US,0.983177,0.981931,0.974731,0.985960,0.997027,0.956782,163562.0


## 5. Evaluation

est_public of stage 1 alone and of the two-stage model on the mock val entities, against v103
and v107 (their public scores are the calibration points of the tight mock).

In [6]:
cmp = pd.DataFrame({
    "v103 (public 0.961)": {"est_public": 0.961, "mock_f05": v103["mock_f05"]},
    "v107 (public 0.966)": {"est_public": EST_TO_BEAT, "mock_f05": v107["mock_f05"]},
    "v110 stage 1 alone": {"est_public": single.loc["all", "est_public"],
                           "mock_f05": single.loc["all", "f_beta"]},
    "v110 two-stage": {"est_public": res.loc["all", "est_public"],
                       "mock_f05": res.loc["all", "f_beta"]},
}).T
cmp["delta_vs_v107"] = cmp["est_public"] - EST_TO_BEAT
for c in ("India", "US"):
    cmp.loc["v110 two-stage", f"est_public_{c}"] = res.loc[c, "est_public"]
cmp.round(4)

,est_public,mock_f05,delta_vs_v107,est_public_India,est_public_US
v103 (public 0.961),0.961,0.9704,-0.004878,NaN,NaN
v107 (public 0.966),0.965878,0.9745,0.0,NaN,NaN
v110 stage 1 alone,0.968134,0.977068,0.002256,NaN,NaN
v110 two-stage,0.973131,0.98165,0.007252,0.9716,0.9747


### Ablation

Stage 2 retrained without M3's columns (stage 1 keeps them) and without the rival columns:
same parts, same early stopping, threshold rule tuned on the tight mock. The full model's
threshold-rule score is the like-for-like reference.

In [7]:
from entity_resolution.features import FEATURE_COLUMNS
from entity_resolution.stacking import COHESION_COLUMNS, RIVAL_COLUMNS
cols_all = list(next(iter(outs.values())).X.columns)
m3_cols = {c for g in M3_GROUPS for c in FEATURE_COLUMNS[g]}
variants = {
    "without M3 columns": [c for c in cols_all if c not in m3_cols],
    "without rival columns": [c for c in cols_all
                              if c not in RIVAL_COLUMNS and c not in COHESION_COLUMNS],
}
rows = {"full, threshold rule": mock_scores(scored, mock, rule_t).loc["all", ["f_beta", "est_public"]]}
t0 = time.time()
for name, cols in variants.items():
    m_ab, _ = fit_stage2(outs, mock, tcfg, columns=cols)
    s_ab, _ = mock_scored(outs, m_ab, mock, tcfg)
    r_ab, _ = tune_mock(s_ab, mock, cfg.grid, fp_weight=FP_WEIGHT)
    rows[name] = mock_scores(s_ab, mock, r_ab).loc["all", ["f_beta", "est_public"]]
    del m_ab, s_ab
ablation = pd.DataFrame(rows).T
ablation["delta"] = ablation["est_public"] - ablation.loc["full, threshold rule", "est_public"]
print(f"ablation {time.time() - t0:.0f} s; {len(m3_cols)} M3 columns of {len(cols_all)}")
ablation.round(4)

ablation 712 s; 23 M3 columns of 96


,f_beta,est_public,delta
"full, threshold rule",0.9815,0.9730,0.0000
without M3 columns,0.9810,0.9725,-0.0005
without rival columns,0.9813,0.9729,-0.0001


## 6. Error analysis

Error counts on the mock val entities for the two-stage model, against v107's.

In [8]:
part = mock.part("val")
matches = apply_rule(scored[isin(scored[C.S1_ID], pd.Index(part.s1[C.ENTITY_ID]))], rule)
counts = {k: len(error_samples(matches, part, k, n=10**9))
          for k in ("false_merge", "missed", "false_singleton", "singleton_merge")}
print("v110:", counts, "\nv107:", v107["metrics"].get("errors_mock"))

v110: {'false_merge': 3169, 'missed': 55068, 'false_singleton': 1619, 'singleton_merge': 334} 
v107: {'v104 plain rule': {'false_merge': 5265, 'missed': 68406, 'false_singleton': 3325, 'singleton_merge': 297}, 'v107 tight rule': {'false_merge': 3522, 'missed': 74223, 'false_singleton': 3154, 'singleton_merge': 362}}


## 7. Log the result

In [9]:
ts = TwoStage(stage1, models, rule, tcfg, table,
              {"fit": fit_info, "stage1_fit": s1_model.fit_info_,
               "filter_report": filter_report.to_dict("index")})
ts.save(ARTIFACTS)
record = {
    "hypothesis": "M3's feature groups, dense blocking and a GPU stage 1 at train-fold "
                  "density lift the two-stage model",
    "feature_groups": list(cfg.feature_groups), "m3_groups": list(M3_GROUPS),
    "blocking_config": asdict(cfg.blocking), "stage1_params": asdict(S1_PARAMS),
    "stage1_cache": str(STAGE1_CACHE), "blocking_key": cfg.blocking.key(),
    "stage1_entities": len(s1_ids), "stage1_rows": manifest["rows"],
    "stage1_fit_info": s1_model.fit_info_, "single_stage_mock_f_beta": single.loc["all", "f_beta"],
    "single_stage_rule": asdict(rule1), "two_stage": tcfg.record(), "rule": asdict(rule),
    "rule_kind": type(rule).__name__, "fp_weight": FP_WEIGHT,
    "mock_f_beta": res.loc["all", "f_beta"], "est_public": res.loc["all", "est_public"],
    "f_tight": res.loc["all", "f_tight"], "single_stage_est_public": single.loc["all", "est_public"],
    **{f"mock_{c}": res.loc[c, "f_beta"] for c in ("India", "US")},
    **{k: res.loc["all", k] for k in ("f_beta_singletons", "f_beta_matched",
                                      "pair_precision", "pair_recall")},
    "cand_recall_val": filter_report.loc["val", "pair_recall"],
    "cands_mean_val": filter_report.loc["val", "candidates_mean"],
    "errors_mock": counts, "ablation": ablation.to_dict("index"), **timings,
    "peak_rss_gb": peak_rss_gb(),
}
DECISION = "KEEP" if record["est_public"] > EST_TO_BEAT + 0.001 else "DROP"
record["decision"] = DECISION
print(f"est_public v107 {EST_TO_BEAT:.4f} -> v110 {record['est_public']:.4f} {DECISION}")
row = log_result(
    EXP_DIR, change=("v107 two-stage + M3 feature groups (idf, token_freq, ctx_idf, "
                     "address_extra), blocking B6 + name+number, GPU stage 1 "
                     f"({N_STAGE1:,} entities), rival features"),
    group="C2", mock_f05=record["mock_f_beta"], cand_recall=record["cand_recall_val"],
    notes=(f"est_public {record['est_public']:.4f}; single stage est "
           f"{record['single_stage_est_public']:.4f}; cands {record['cands_mean_val']:.1f}/S1; "
           f"stage 2 without M3 columns "
           f"{ablation.loc['without M3 columns', 'delta']:+.4f}; blocking {cfg.blocking.key()}"),
    metrics=record, owner="M1", parent="v107", decision=DECISION)
row

est_public v107 0.9659 -> v110 0.9731 KEEP


{'version': 'v110',
 'date': '2026-09-26',
 'group': 'C2',
 'change': 'v107 two-stage + M3 feature groups (idf, token_freq, ctx_idf, address_extra), blocking B6 + name+number, GPU stage 1 (110,000 entities), rival features',
 'local_f05': '',
 'mock_f05': '0.9817',
 'cand_recall': '0.9788',
 'public_f05': '',
 'commit': 'c9ec108',
 'notes': 'est_public 0.9731; single stage est 0.9681; cands 4.4/S1; stage 2 without M3 columns -0.0005; blocking db1f33c4',
 'owner': 'M1',
 'parent': 'v107',
 'decision': 'KEEP'}

## 8. Conclusion

Written after the run from the numbers above.

## 9. Test inference

`run_test_two_stage` with this stage 1 and stage 2 (stage-1 outputs cached per stage 1,
blocking and filter), both validators, files kept in `submissions/v110/`.

In [10]:
t0 = time.time()
match_path, cand_path, s1n_test, test_matches, test_summary = run_test_two_stage(
    cfg, ts, cache_dir=STAGE1_CACHE / "test")
print(f"run_test {time.time() - t0:.0f} s")
country_of = s1n_test.set_index(C.ENTITY_ID)[C.COUNTRY]
n_s1 = s1n_test.groupby(C.COUNTRY).size()
by = test_matches[C.S1_ID].map(country_of)
display(pd.DataFrame({
    "s1": n_s1,
    "cands_per_s1": test_summary["n_cands"].groupby(test_summary.index.map(country_of)).sum() / n_s1,
    "matched_share": test_matches.groupby(by)[C.S1_ID].nunique() / n_s1,
    "matches_per_s1": test_matches.groupby(by).size() / n_s1}))
dest = C.ROOT / "submissions" / "v110"
dest.mkdir(parents=True, exist_ok=True)
for p in (match_path, cand_path):
    shutil.copy2(p, dest / p.name)
out = subprocess.run([sys.executable, "-m", "entity_resolution.submission", "--output-dir",
                      str(C.OUTPUT), "--check-ids"], capture_output=True, text=True)
print(out.stdout[-2000:], out.stderr[-2000:])
out = subprocess.run([sys.executable, str(C.OFFICIAL_VALIDATOR), "--matching", str(match_path),
                      "--candidate", str(cand_path), "--test-dir", str(C.DATASET / "test")],
                     capture_output=True, text=True)
print(out.stdout[-3000:], out.stderr[-2000:])
print(f"notebook total {time.time() - t_start:.0f} s, peak RSS {peak_rss_gb()} GB")

run_test 5434 s


,s1,cands_per_s1,matched_share,matches_per_s1
France,259452,6.128687,0.948075,3.354910
India,809986,4.801342,0.940737,3.296409
US,663106,5.036118,0.942726,3.428919


PASS
 


ML Challenge 2026 — submission validator
  test dir: /home/suryaguru/StudioProjects/aws/business_entity_resolution/dataset/student_resource/dataset/test
  required S1 entities: 1732544
  matching_results.tsv: 1732544 rows (99453 empty, 1633091 non-empty).
  candidate_pairs.tsv: 1732544 rows (24603 empty, 1707941 non-empty).

PASS — no blocking issues found. Safe to submit.
 
notebook total 11997 s, peak RSS 7.68 GB
